# 05 - Model Training & Evaluation

Train a baseline classifier to predict whether a property is
near the beach, then evaluate on the held-out test split.

In [ ]:
# Make the project `src` package importable from the notebooks/ dir.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_dirs()

In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

train = pd.read_parquet(config.TRAIN_FILE)
test = pd.read_parquet(config.TEST_FILE)
target = config.TARGET_COLUMN

## Prepare feature matrices

In [ ]:
feature_cols = [c for c in train.columns if c != target]
# Use only numeric features for this baseline; encode categoricals as needed.
feature_cols = train[feature_cols].select_dtypes(include="number").columns.tolist()

X_train, y_train = train[feature_cols], train[target]
X_test, y_test = test[feature_cols], test[target]
print(f"Using {len(feature_cols)} features")

## Train baseline model

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## Evaluate

In [ ]:
preds = model.predict(X_test)
print(classification_report(y_test, preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, preds))

## Feature importance

In [ ]:
importances = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
)
importances

## Persist model

In [ ]:
model_path = config.MODELS_DIR / "beach_predictor.joblib"
joblib.dump(model, model_path)
print(f"Saved model -> {model_path}")